# Project 4: hoeveel van Suriname is nog bos?

**De vraag.** Hoeveel van het land van Suriname is nog bos, en hoe snel verandert dat?

**Voor wie het is.** Een docent aardrijkskunde op een middelbare school in Suriname, en de klas van die docent. Zij krijgen een grafiekenblad van één pagina voor een les: een eerlijk beeld van hoeveel van hun land bos is en hoe snel het verdwijnt, een misleidende grafiek van dezelfde getallen met zijn stempel, en een korte uitleg van hoe je de twee uit elkaar houdt.

**De data.** Twee reeksen van de Wereldbank voor Suriname, allebei overgenomen van de Voedsel- en Landbouworganisatie van de Verenigde Naties (FAO): het bosoppervlak als percentage van het landoppervlak (indicator AG.LND.FRST.ZS) en het bosoppervlak in vierkante kilometer (indicator AG.LND.FRST.K2). De Wereldbank heeft beide reeksen voor het laatst bijgewerkt op 2026-10-08. Het cursusbestand zet ze naast elkaar in één tabel, met één rij per jaar.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator. Deze week controleert de datacontroleur of de twee kolommen hetzelfde verhaal vertellen, en toetst de AI-controleur de grafiek en de beschrijving van de AI aan de getallen.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, het grafiekenblad voor de aardrijkskundeklas (de eerlijke en de misleidende grafiek naast elkaar, de grafiek van hoe snel het bos verdwijnt, en ongeveer 200 woorden), en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of bouwt hem rechtstreeks bij de Wereldbank als het bestand er niet is.

In [ ]:
import json, urllib.request
import pandas as pd

URL = "https://api.worldbank.org/v2/country/SUR/indicator/{}?format=json&date=1990:2025&per_page=100"
try:
    forest = pd.read_csv("../data/week04-forest.csv")
except FileNotFoundError:
    # geen cursusbestand: bouw dezelfde tabel uit de twee reeksen van de Wereldbank
    columns = {"AG.LND.FRST.ZS": "forest_pct", "AG.LND.FRST.K2": "forest_km2"}
    parts = []
    for code, name in columns.items():
        raw = json.load(urllib.request.urlopen(URL.format(code)))[1]
        parts.append(pd.Series({int(r["date"]): r["value"] for r in raw}, name=name, dtype=float))
    forest = pd.concat(parts, axis=1).dropna(how="all").rename_axis("year").sort_index().reset_index()
forest

Ook deze cel krijg je cadeau. Hij maakt twee korte namen voor de kolommen, met het jaar als index, en neemt de overdrijvingsfactor uit het vaardighedennotebook over.

In [ ]:
import matplotlib.pyplot as plt

pct = forest.set_index("year")["forest_pct"]     # bosoppervlak, procent van het landoppervlak
km2 = forest.set_index("year")["forest_km2"]     # bosoppervlak, vierkante kilometer

def exaggeration(base, other, bottom):
    """Hoeveel keer groter een grafiek de verandering van base naar other laat lijken dan ze is.
    bottom is de waarde waar de as begint. Een grafiek vanaf nul heeft factor 1."""
    real = (other - base) / base
    drawn = (other - base) / (base - bottom)
    return drawn / real

## TODO 1: beschrijf de tabel

Druk het aantal rijen, het eerste en laatste jaar en de ontbrekende waarden per kolom af. Sla daarna in `last_pct_year` en `last_km2_year` het laatste jaar op dat in elke kolom een waarde heeft. De twee kolommen hoeven niet in hetzelfde jaar te eindigen.

In [ ]:
# TODO 1
last_pct_year = None
last_km2_year = None

In [ ]:
# CHECK 1
assert last_pct_year is not None and last_km2_year is not None, "sla beide jaren op"
assert last_pct_year == forest.loc[forest["forest_pct"].notna(), "year"].max()
assert last_km2_year == forest.loc[forest["forest_km2"].notna(), "year"].max()
print("CHECK 1 geslaagd: procent tot", last_pct_year, "en vierkante kilometer tot", last_km2_year)

## TODO 2: de eerlijke grafiek van het niveau

Maak een lijngrafiek van `forest_pct` per jaar, met de y-as van 0 tot 100 procent. Geef hem een titel die de bewering uitspreekt, benoem beide assen met hun eenheden, en noem de Wereldbank als bron in een notitie op de grafiek. Sla de assen op in `honest_ax`.

In [ ]:
# TODO 2
honest_ax = None

In [ ]:
# CHECK 2
assert honest_ax is not None, "honest_ax is nog leeg"
lo, hi = honest_ax.get_ylim()
assert lo == 0 and hi >= 100, "de y-as van de eerlijke grafiek loopt van 0 tot 100 procent"
assert honest_ax.get_title().strip() and honest_ax.get_ylabel().strip(), "geef de grafiek een titel met een bewering en een label op de y-as"
print("CHECK 2 geslaagd: de as loopt van", lo, "tot", hi)

## TODO 3: meet de verandering

Vergelijk gelijk met gelijk: sla in `end_year` het laatste jaar op dat beide kolommen hebben. Sla in `peak_year` het jaar met het meeste bos op. Sla daarna op:

- `points_lost`: de daling van `forest_pct` van `peak_year` tot `end_year`, in procentpunten;
- `km2_lost`: de daling van `forest_km2` over dezelfde jaren, in vierkante kilometer;
- `share_lost`: `km2_lost` als percentage van het bos in `peak_year`;
- `points_since_1990`: de daling van `forest_pct` van 1990 tot `end_year`, in procentpunten.

De datacontroleur berekent `share_lost` daarna op een tweede manier, uit de procentkolom, en bevestigt dat de twee kolommen overeenkomen. Zeg op het grafiekenblad welk beginjaar je gebruikt en waarom.

In [ ]:
# TODO 3
end_year = None
peak_year = None
points_lost = None
km2_lost = None
share_lost = None
points_since_1990 = None

In [ ]:
# CHECK 3
assert None not in (end_year, peak_year, points_lost, km2_lost, share_lost, points_since_1990), "vul alle zes in"
assert end_year == forest.dropna()["year"].max(), "end_year is het laatste jaar met een waarde in beide kolommen"
assert peak_year == km2.idxmax()
assert abs(points_lost - (pct[peak_year] - pct[end_year])) < 0.001
assert abs(km2_lost - (km2[peak_year] - km2[end_year])) < 0.1
assert abs(share_lost - km2_lost / km2[peak_year] * 100) < 0.001
assert abs(points_since_1990 - (pct[1990] - pct[end_year])) < 0.001
print("CHECK 3 geslaagd:", peak_year, "tot", end_year, round(points_lost, 3), "punten,", round(km2_lost, 1), "km2,", round(share_lost, 3), "procent")
print("verloren aandeel, uit de procentkolom:", round(points_lost / pct[peak_year] * 100, 3), "procent")

## TODO 4: hoe snel?

Druk de jaarlijkse verandering in vierkante kilometer af met `km2.diff()` en lees die. De verandering blijft een aantal jaren achter elkaar gelijk en springt dan naar een nieuwe waarde. Schrijf in je AI-logboek wat dat patroon je vertelt over hoe de tussenliggende jaren zijn gemaakt.

Sla daarna in `loss_per_year` het gemiddelde aantal vierkante kilometer op dat per jaar verloren ging in elk van de vier periodes hieronder, als een Series met de labels `"2000-2010"`, `"2010-2015"`, `"2015-2020"` en `"2020-2025"`. Sla in `speedup` op hoeveel keer sneller het verlies in 2020-2025 ging dan in 2000-2010.

Maak van `loss_per_year` een staafgrafiek met staven vanaf nul, een titel met een bewering en benoemde assen, en sla de assen op in `speed_ax`.

In [ ]:
# TODO 4
periods = [(2000, 2010), (2010, 2015), (2015, 2020), (2020, 2025)]
loss_per_year = None
speedup = None
speed_ax = None

In [ ]:
# CHECK 4
assert loss_per_year is not None and speedup is not None and speed_ax is not None, "vul alle drie in"
expected = pd.Series({f"{a}-{b}": (km2[a] - km2[b]) / (b - a) for a, b in [(2000, 2010), (2010, 2015), (2015, 2020), (2020, 2025)]})
assert [str(i) for i in loss_per_year.index] == list(expected.index), "geef de periodes de labels 2000-2010, 2010-2015, 2015-2020 en 2020-2025"
assert (abs(loss_per_year.values - expected.values) < 0.1).all(), "verliezen zijn positieve vierkante kilometers per jaar"
assert abs(speedup - expected["2020-2025"] / expected["2000-2010"]) < 0.01
assert speed_ax.get_ylim()[0] == 0, "de staven beginnen bij nul"
print("CHECK 4 geslaagd:", [round(float(v), 1) for v in expected.values], "km2 per jaar; versnelling", round(speedup, 2))

## TODO 5: de misleidende grafiek, met opzet gebouwd

Teken de eerlijke grafiek en een misleidende grafiek van dezelfde reeks `forest_pct` naast elkaar, met `plt.subplots(1, 2)`. Snijd in de misleidende grafiek de y-as zo af dat de daling van `peak_year` tot `end_year` minstens 20 keer groter lijkt dan ze is, houd elk punt op de grafiek, en geef hem een dramatische titel. Zet er met `ax.text` de woorden *misleidend, met opzet* op als stempel.

Sla de assen van de misleidende grafiek op in `cut_ax` en de overdrijvingsfactor in `cut_factor`. De misleidende grafiek verschijnt alleen naast de eerlijke, met zijn stempel.

In [ ]:
# TODO 5
cut_ax = None
cut_factor = None

In [ ]:
# CHECK 5
assert cut_ax is not None and cut_factor is not None, "vul cut_ax en cut_factor in"
stamp = " ".join(t.get_text().lower() for t in cut_ax.texts)
assert "misleading" in stamp or "misleidend" in stamp, "zet de woorden misleidend, met opzet als stempel op de grafiek"
bottom = cut_ax.get_ylim()[0]
assert bottom < forest["forest_pct"].min(), "houd elk punt op de grafiek"
assert abs(cut_factor - exaggeration(pct[peak_year], pct[end_year], bottom)) < 0.01
assert cut_factor >= 20, "snijd de as dichter bij de data af: de factor moet minstens 20 zijn"
print("CHECK 5 geslaagd: de as begint bij", round(bottom, 2), "en de factor is", round(cut_factor, 1))

## TODO 6: de AI-controle

Vraag je AI-assistent, met deze woorden: *"Mijn tabel `forest` heeft de kolommen year, forest_pct (bosoppervlak in procent van het landoppervlak) en forest_km2. Schrijf matplotlib-code voor een lijngrafiek van forest_pct per jaar."*

1. Plak de code in de cel hieronder en voer hem uit. Zet vlak vóór `plt.show()` de regel `ai_ax = plt.gca()`.
2. Sla de grafiek op als afbeelding, geef die aan de AI en vraag: *"Wat laat deze grafiek in twee zinnen zien over het bos van Suriname?"* Schrijf het antwoord woord voor woord in jullie AI-logboek.
3. Sla in `ai_factor` de overdrijvingsfactor van de grafiek van de AI op voor de daling van `peak_year` tot `end_year`, met `ai_ax.get_ylim()[0]` als onderkant.
4. Toets in het AI-logboek elk woord in de beschrijving van de AI dat zegt hoe groot of hoe snel, aan `share_lost` en `speedup`.

Als de grafiek van de AI bij nul begint, is de factor 1, en dat noteer je ook in het logboek.

In [ ]:
# TODO 6
ai_ax = None
ai_factor = None
# Plak de code van de AI hieronder. Zet vlak vóór plt.show() de regel: ai_ax = plt.gca()

In [ ]:
# CHECK 6
assert ai_ax is not None and ai_factor is not None, "vul ai_ax en ai_factor in"
ai_bottom = ai_ax.get_ylim()[0]
assert abs(ai_factor - exaggeration(pct[peak_year], pct[end_year], ai_bottom)) < 0.01
print("CHECK 6 geslaagd: de as van de AI begint bij", round(ai_bottom, 2), "dus de grafiek laat de daling", round(ai_factor, 1), "keer groter lijken dan ze is")
print("De echte daling van", peak_year, "tot", end_year, "is", round(share_lost, 2), "procent van het bos.")

## TODO 7: het grafiekenblad voor een aardrijkskundeklas

Maak een grafiekenblad van één pagina dat een docent aardrijkskunde op een middelbare school in Suriname aan een klas kan geven. Zet er drie grafieken op: de eerlijke grafiek uit opdracht 2, de snelheidsgrafiek uit opdracht 4, en de misleidende grafiek met stempel naast zijn eerlijke tweeling uit opdracht 5.

Schrijf in een tekstcel hieronder de tekst van het blad: ongeveer 200 woorden in eenvoudige taal voor leerlingen van de middelbare school, over vier punten.

- Hoeveel van het land bos is, en de verandering in procentpunten en in procent, met het beginjaar erbij.
- Hoe snel het verlies nu gaat vergeleken met 2000-2010, in vierkante kilometer per jaar.
- Hoe de misleidende grafiek het oog bedriegt, met de overdrijvingsfactor, en één vraag die een leerling bij elke grafiek kan stellen om de truc te ontdekken.
- Eén ding dat de tabel de klas niet kan vertellen, zoals waarom het bos krimpt of waar.

Leg elk begrip uit dat de klas misschien niet kent, zoals procentpunt. De datacontroleur leest het blad vóór vrijdag na tegen het notebook.